# Despliegue y promoción

Actualiza Model Serving, ejecuta smoke test y recién después promueve `Champion`.

In [ ]:
import os

import mlflow
from databricks.sdk import WorkspaceClient
from mlflow.tracking import MlflowClient

from iris_mlflow_utils import (
    build_config,
    build_deployment_config,
    load_dataset_from_spark,
    promote_champion,
    update_serving_endpoint,
    wait_for_endpoint_ready,
)

dbutils.widgets.text('model_name', 'workspace.default.iris_classifier')
dbutils.widgets.text('model_version', '')
model_name = dbutils.widgets.get('model_name')
model_version = dbutils.widgets.get('model_version')
if not model_version:
    raise ValueError('model_version es obligatorio')
deployment_config = build_deployment_config()
training_config = build_config(model_slug='random_forest')
registry = MlflowClient(registry_uri='databricks-uc')
version = registry.get_model_version(model_name, model_version)
if version.tags.get('evaluation_status') != 'passed':
    raise RuntimeError('La versión no superó evaluación.')
if version.tags.get('approval_status') != 'approved':
    raise RuntimeError('La versión no tiene aprobación manual.')
workspace = WorkspaceClient()
update_serving_endpoint(
    workspace, endpoint_name=deployment_config.endpoint_name,
    model_name=model_name, model_version=model_version
)
wait_for_endpoint_ready(
    workspace, deployment_config.endpoint_name,
    timeout_seconds=deployment_config.serving_timeout_seconds,
    poll_seconds=deployment_config.serving_poll_seconds,
)


In [ ]:
dataset = load_dataset_from_spark(
    spark, table_name=training_config.feature_table,
    table_version=training_config.feature_table_version,
    target_column=training_config.target_column,
)
sample = dataset.features.head(deployment_config.smoke_test_rows)
response = workspace.serving_endpoints.query(
    name=deployment_config.endpoint_name,
    dataframe_split={
        'columns': [str(column) for column in sample.columns],
        'data': sample.values.tolist(),
    },
)
predictions = getattr(response, 'predictions', None)
if predictions is None and isinstance(response, dict):
    predictions = response.get('predictions')
if not isinstance(predictions, list) or len(predictions) != len(sample):
    raise RuntimeError(f'Smoke test inválido: {response!r}')
promote_champion(
    registry, model_name=model_name, model_version=model_version,
    champion_alias=deployment_config.champion_alias
)
registry.set_model_version_tag(model_name, model_version, 'deployment_status', 'deployed')
dbutils.jobs.taskValues.set(key='deployment_status', value='deployed')
print({'endpoint': deployment_config.endpoint_name, 'model_version': model_version, 'status': 'deployed'})
